# Shared Model Setup

This notebook defines the shared setup that every model notebook must use:
the same input dataset, train/test split, preprocessing pipeline, random
state, cross-validation strategy and evaluation metrics.

Model tuning and model selection should use the training set with
cross-validation. The final test set should be used only for final model
evaluation.

In [11]:
from pathlib import Path
import sys

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/processed/hotel_booking_features.csv").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


from scripts.model_evaluation_utils import (
    FEATURE_DATA_PATH,
    RANDOM_STATE,
    TEST_SIZE,
    CV_SPLITS,
    COMMON_SCORING,
    load_feature_dataset,
    split_feature_dataset,
    class_distribution,
    build_preprocessor,
    get_cv,
)

## 1. Load Final Feature Dataset

In [12]:
data = load_feature_dataset()
print(f"Dataset path: {FEATURE_DATA_PATH.relative_to(ROOT)}")
print(f"Dataset shape: {data.shape[0]:,} rows x {data.shape[1]} columns")
print(f"Target values: {sorted(data['is_canceled'].unique())}")
data.head()

Dataset path: data\processed\hotel_booking_features.csv
Dataset shape: 119,205 rows x 33 columns
Target values: [np.int64(0), np.int64(1)]


,is_canceled,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_stay,total_guests,has_children,has_babies,previous_total_bookings,previous_cancellation_rate
0,0,Resort Hotel,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,2,0,0,0,0.0
1,0,Resort Hotel,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,2,0,0,0,0.0
2,0,Resort Hotel,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,1,0,0,0,0.0
3,0,Resort Hotel,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,1,0,0,0,0.0
4,0,Resort Hotel,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,2,2,0,0,0,0.0


## 2. Shared Train/Test Split

In [13]:
prepared = split_feature_dataset(data)

print(f"Random state: {RANDOM_STATE}")
print(f"Test size: {TEST_SIZE:.0%}")
print(f"CV splits: {CV_SPLITS}")
print(f"Training rows: {len(prepared.X_train):,}")
print(f"Test rows: {len(prepared.X_test):,}")

print("\nTraining class distribution")
display(class_distribution(prepared.y_train))

print("\nTest class distribution")
display(class_distribution(prepared.y_test))

Random state: 42
Test size: 20%
CV splits: 5
Training rows: 95,364
Test rows: 23,841

Training class distribution


,class,count,percent
0,0,60008,62.9252
1,1,35356,37.0748



Test class distribution


,class,count,percent
0,0,15002,62.9252
1,1,8839,37.0748


## 3. Shared Preprocessing

Numeric fields are robust-scaled. Low-cardinality categorical fields are
one-hot encoded. High-cardinality fields (`country`, `agent`, `company`) use
infrequent-category grouping learned from the training data only.

In [14]:
preprocessor = build_preprocessor(prepared.X_train)
transformed_sample = preprocessor.fit_transform(prepared.X_train.head(100))

print(preprocessor)
print(f"Sample transformed shape: {transformed_sample.shape}")

ColumnTransformer(sparse_threshold=1.0,
                  transformers=[('numeric', RobustScaler(),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'children',
                                  'babies', 'previous_cancellations',
                                  'previous_bookings_not_canceled', 'adr',
                                  'required_car_parking_spaces',
                                  'total_of_s...
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['hotel', 'arrival_date_month', 'meal',
                                  'market_segment', 'distribution_channel',
                                  'reserved_room_type', 'deposit_type',
             

## 4. Cross-Validation and Metrics

In [15]:
cv = get_cv()
print(cv)
print("Scoring metrics:")
for name, scorer in COMMON_SCORING.items():
    print(f"- {name}: {scorer}")

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Scoring metrics:
- accuracy: accuracy
- precision: precision
- recall: recall
- f1: f1
- roc_auc: roc_auc
- pr_auc: average_precision
